<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_03_5_pandas_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks

**Module 3: Working with Tabular Data**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 3 Material

Main video lecture:

- Part 3.1 Should Neural Networks be Used for Tabular Data [[Video]]() [[Notebook]](t81_558_class_03_1_python_pandas.ipynb)
- Part 3.2 Encoding Categorical Values in Pandas [[Video]]() [[Notebook]](t81_558_class_03_2_pandas_cat.ipynb)
- Part 3.3 Grouping, Sorting and Shuffling [[Video]]() [[Notebook]](t81_558_class_03_3_pandas_grouping.ipynb)
- Part 3.4 Pandas Functional [[Video]]() [[Notebook]](t81_558_class_03_4_pandas_functional.ipynb)
- **Part 3.5 Feature Engineering** [[Video]]() [[Notebook]](t81_558_class_03_5_pandas_features.ipynb)

# Google Colab Instructions

The following code ensures that Google Colab is running and maps Google Drive if needed.

In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google Colab")
except ImportError:
    COLAB = False
    print("Note: not using Google Colab")

Note: using Google Colab


# Part 3.5: Feature Engineering

Feature engineering is an essential part of machine learning.  For now, we will manually engineer features.  However, later in this course, we will see some techniques for automatic feature engineering.  

## Calculated Fields

It is possible to add new fields to the data frame that your program calculates from the other fields.  We can create a new column that gives the weight in kilograms.  The equation to calculate a metric weight, given weight in pounds, is:

$$ m_{(kg)} = m_{(lb)} \times 0.45359237 $$

The following Python code performs this transformation:

In [2]:
import os
import pandas as pd

df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=['NA', '?'])

df.insert(1, 'weight_kg', (df['weight'] * 0.45359237).astype(int))
pd.set_option('display.max_columns', 6)
pd.set_option('display.max_rows', 5)
df

,mpg,weight_kg,cylinders,...,year,origin,name
0,18.0,1589,8,...,70,1,chevrolet chevelle malibu
1,15.0,1675,8,...,70,1,buick skylark 320
...,...,...,...,...,...,...,...
396,28.0,1190,4,...,82,1,ford ranger
397,31.0,1233,4,...,82,1,chevy s-10


## Google API Keys

Sometimes you will use external APIs to obtain data.  The following examples demonstrate how to use Google API keys to encode addresses for neural networks.  To use these, you will need your own Google API key.  The key I have below is not a real key; you need to put your own there.  Google will ask for a credit card, but there will be no actual charge unless you make a large number of lookups.  YOU ARE NOT required to get a Google API key for this class; this only shows you how.  If you want to get a Google API key, visit this site and obtain one for **geocode**.

You can obtain your key from this link: [Google API Keys](https://developers.google.com/maps/documentation/embed/get-api-key).

In [3]:
import os

try:
    from google.colab import userdata
    GOOGLE_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_KEY = None

# Fallback to environment variable if not in Colab or not set
if not GOOGLE_KEY:
    GOOGLE_KEY = os.environ.get("GOOGLE_API_KEY")

# Fail fast if still missing
if not GOOGLE_KEY:
    raise RuntimeError(
        "GOOGLE_API_KEY not found. Set it in Colab Secrets or as an environment variable."
    )

## Other Examples: Dealing with Addresses

Addresses can be difficult to encode into a neural network.  There are many approaches, and you must consider how to transform the address into something more meaningful.  Map coordinates can be a good approach; [latitude and longitude](https://en.wikipedia.org/wiki/Geographic_coordinate_system) can also be a useful encoding.  Thanks to the power of the Internet, it is relatively easy to convert an address into its latitude and longitude.  The following code determines the coordinates of [Washington University](https://wustl.edu/):

In [4]:
import requests

address = "1 Brookings Dr, St. Louis, MO 63130"

url = "https://maps.googleapis.com/maps/api/geocode/json"

params = {
    "key": GOOGLE_KEY,
    "address": address,
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()  # Raises HTTPError for bad responses

resp_json_payload = response.json()

if resp_json_payload.get("error_message"):
    print(resp_json_payload["error_message"])
elif resp_json_payload.get("results"):
    location = resp_json_payload["results"][0]["geometry"]["location"]
    print(location)
else:
    print("No results returned.")

{'lat': 38.6493165, 'lng': -90.310654}


They might not be overly helpful if you feed latitude and longitude into the neural network as two features.  These two values would allow your neural network to cluster locations on a map.  Sometimes cluster locations on a map can be useful.  Figure 2.SMK shows the percentage of the population that smokes in the USA by state.

**Figure 2.SMK: Smokers by State**
![Smokers by State](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_6_smokers.png "Smokers by State")

The map above shows that certain behaviors, such as smoking, can be clustered by global region.

However, often you will want to transform the coordinates into distances.  It is reasonably easy to estimate the distance between any two points on Earth by using the [great circle distance](https://en.wikipedia.org/wiki/Great-circle_distance) between any two points on a sphere:

The following code implements this formula:

$$ \Delta\sigma=\arccos\bigl(\sin\phi_1\cdot\sin\phi_2+\cos\phi_1\cdot\cos\phi_2\cdot\cos(\Delta\lambda)\bigr) $$

$$ d = r \, \Delta\sigma $$

In [5]:
from math import atan2, cos, radians, sin, sqrt

import requests

GOOGLE_GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"


def distance_lat_lng(lat1, lng1, lat2, lng2):
    """Calculate the great-circle distance between two points in kilometers."""
    earth_radius_km = 6371.0

    lat1 = radians(lat1)
    lng1 = radians(lng1)
    lat2 = radians(lat2)
    lng2 = radians(lng2)

    dlng = lng2 - lng1
    dlat = lat2 - lat1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlng / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return earth_radius_km * c


def lookup_lat_lng(address):
    """Look up the latitude and longitude for an address using Google Geocoding."""
    params = {
        "key": GOOGLE_KEY,
        "address": address,
    }

    response = requests.get(GOOGLE_GEOCODE_URL, params=params, timeout=10)
    response.raise_for_status()

    payload = response.json()
    status = payload.get("status")

    if status != "OK":
        error_message = payload.get("error_message", "No additional error message.")
        raise ValueError(f"Google API error for '{address}': {status}. {error_message}")

    location = payload["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]


address1 = "1 Brookings Dr, St. Louis, MO 63130"
address2 = "3301 College Ave, Fort Lauderdale, FL 33314"

lat1, lng1 = lookup_lat_lng(address1)
lat2, lng2 = lookup_lat_lng(address2)

distance = distance_lat_lng(lat1, lng1, lat2, lng2)

print(f"Distance, St. Louis, MO to Ft. Lauderdale, FL: {distance:.2f} km")

Distance, St. Louis, MO to Ft. Lauderdale, FL: 1685.17 km


Distances can be a useful means to encode addresses.  It would help if you considered what distance might be helpful for your dataset.  Consider:

* Distance to a major metropolitan area
* Distance to a competitor
* Distance to a distribution center
* Distance to a retail outlet

The following code calculates the distance between 10 universities and Washington University in St. Louis:

In [6]:
# Encoding other universities by their distance to Washington University

schools = [
    ["Princeton University, Princeton, NJ 08544", 'Princeton'],
    ["Massachusetts Hall, Cambridge, MA 02138", 'Harvard'],
    ["5801 S Ellis Ave, Chicago, IL 60637", 'University of Chicago'],
    ["Yale, New Haven, CT 06520", 'Yale'],
    ["116th St & Broadway, New York, NY 10027", 'Columbia University'],
    ["450 Serra Mall, Stanford, CA 94305", 'Stanford'],
    ["77 Massachusetts Ave, Cambridge, MA 02139", 'MIT'],
    ["Duke University, Durham, NC 27708", 'Duke University'],
    ["University of Pennsylvania, Philadelphia, PA 19104",
         'University of Pennsylvania'],
    ["Johns Hopkins University, Baltimore, MD 21218", 'Johns Hopkins']
]

lat1, lng1 = lookup_lat_lng("1 Brookings Dr, St. Louis, MO 63130")

for address, name in schools:
    lat2,lng2 = lookup_lat_lng(address)
    dist = distance_lat_lng(lat1,lng1,lat2,lng2)
    print("School '{}', distance to wustl is: {}".format(name,dist))

School 'Princeton', distance to wustl is: 1354.5120314201838
School 'Harvard', distance to wustl is: 1670.5247016432486
School 'University of Chicago', distance to wustl is: 418.1085638397287
School 'Yale', distance to wustl is: 1508.181441730162
School 'Columbia University', distance to wustl is: 1416.236310531565
School 'Stanford', distance to wustl is: 2779.366077887927
School 'MIT', distance to wustl is: 1672.3392252113879
School 'Duke University', distance to wustl is: 1046.9799384590006
School 'University of Pennsylvania', distance to wustl is: 1307.4316224473555
School 'Johns Hopkins', distance to wustl is: 1184.4842010298366
